---

## 🎛️  Deep Dive into Generation Parameters

Understanding generation parameters is crucial for getting the **exact type of output** you want from LLMs. Let's explore each parameter in detail!

---

## 📊 Overview: All Key Parameters

| Parameter | Range | What It Controls | Best For |
|-----------|-------|------------------|----------|
| **Temperature** | 0.0 - 2.0 | Randomness/Creativity | Balancing predictability vs creativity |
| **Top-k** | 1 - 100+ | Vocabulary diversity | Limiting word choices |
| **Top-p (Nucleus)** | 0.0 - 1.0 | Dynamic vocabulary filtering | Natural, diverse text |
| **Max Output Tokens** | 1 - Model Max | Response length | Controlling verbosity |
| **Stop Sequences** | Text strings | When to stop generating | Custom endpoints |
| **Frequency Penalty** | -2.0 - 2.0 | Repetition reduction | Avoiding repeated words |
| **Presence Penalty** | -2.0 - 2.0 | Topic diversity | Encouraging new topics |

---

## 🌡️ 1. Temperature: The Creativity Dial

### What It Does

**Temperature** controls the **randomness** of predictions by scaling the probability distribution.

**Simple Analogy:** Think of it like a thermostat for creativity:
- **Cold (Low Temperature):** Focused, predictable, "safe" choices
- **Hot (High Temperature):** Wild, creative, unpredictable choices

---

### How Temperature Works (Technical)

**The Math Behind It:**

When the model predicts the next token, it generates a probability for each possible token:

```
Before Temperature Scaling:
Token Probabilities:
- "Paris" → 70%
- "London" → 20%
- "Berlin" → 8%
- "Rome" → 2%
```

**With Temperature = 1.0 (Normal):**
- Probabilities stay the same
- Model uses original distribution

**With Temperature = 0.1 (Very Low/Cold):**
```python
# Probabilities become MORE extreme (sharper)
adjusted_prob = original_prob ^ (1/temperature)

Result:
- "Paris" → 99.9%  ← Dominant choice!
- "London" → 0.05%
- "Berlin" → 0.03%
- "Rome" → 0.02%
```
Almost always picks "Paris"!

**With Temperature = 2.0 (High/Hot):**
```python
Result:
- "Paris" → 45%   ← Less dominant
- "London" → 30%
- "Berlin" → 15%
- "Rome" → 10%
```
More likely to pick alternatives!

---

### Temperature Guidelines

| Temperature | Behavior | Use Cases | Example Output |
|-------------|----------|-----------|----------------|
| **0.0** | Deterministic (always same) | Unit tests, exact answers | "2 + 2 = 4" |
| **0.1 - 0.3** | Very focused, consistent | Code generation, factual Q&A, data extraction | Consistent, accurate code |
| **0.4 - 0.6** | Balanced | General conversation, tutoring | Natural but reliable responses |
| **0.7 - 0.9** | Creative, varied | Creative writing, brainstorming | Unique story ideas |
| **1.0 - 1.5** | Very creative | Poetry, artistic content | Experimental, novel text |
| **1.6 - 2.0** | Extremely random | Experimental generation | Can be incoherent |

---

### Temperature Examples

**Prompt:** "Write a tagline for a coffee shop."

```
Temperature 0.1:
"Fresh coffee, freshly brewed every day."
(Run again: Same output!)

Temperature 0.5:
"Where every cup tells a story."
(Run again: Similar but slightly different)

Temperature 1.0:
"Awaken your senses, one sip at a time."
(Run again: Could be completely different)

Temperature 1.5:
"Caffeine dreams dance in moonlit steam."
(Run again: Very different, poetic, unusual)
```

---

## 🎯 2. Top-k Sampling: Limiting the Vocabulary

### What It Does

**Top-k** limits the model to choosing from only the **k most likely tokens** at each step.

**Simple Analogy:** Like only looking at the top k options on a menu instead of the entire menu.

---

### How Top-k Works

**Example: Predicting next word after "The weather is"**

**Full Vocabulary (50,000 tokens available):**
```
Probabilities:
1. "sunny" → 25%
2. "nice" → 20%
3. "beautiful" → 15%
4. "pleasant" → 10%
5. "good" → 8%
6. "great" → 7%
7. "cloudy" → 5%
8. "rainy" → 4%
9. "terrible" → 3%
10. "awful" → 2%
... (49,990 more tokens with tiny probabilities)
```

**With Top-k = 5:**
```
Model only considers:
1. "sunny" → 25% → renormalized to 32%
2. "nice" → 20% → renormalized to 26%
3. "beautiful" → 15% → renormalized to 19%
4. "pleasant" → 10% → renormalized to 13%
5. "good" → 8% → renormalized to 10%

All other 49,995 tokens are ignored!
```

---

### Top-k Guidelines

| Top-k Value | Behavior | Use Cases |
|-------------|----------|-----------|
| **1** | Greedy (always picks #1) | Maximum consistency, deterministic |
| **5-10** | Conservative | Formal writing, technical docs |
| **20-40** | Balanced | General conversation |
| **50-100** | Diverse | Creative writing |
| **Unlimited** | All tokens considered | Maximum creativity (can be chaotic) |

---

### Top-k vs Temperature

**Key Difference:**
- **Temperature:** Changes the probability distribution (shape)
- **Top-k:** Cuts off low-probability choices (filter)

**They work together!**
```
Step 1: Apply Top-k (filter to top k tokens)
Step 2: Apply Temperature (adjust probabilities)
Step 3: Sample from resulting distribution
```

---

## 🎪 3. Top-p (Nucleus Sampling): Dynamic Filtering

### What It Does

**Top-p** (also called Nucleus Sampling) dynamically selects the **smallest set of tokens whose cumulative probability exceeds p**.

**Simple Analogy:** Like filling a bucket with the most likely words until it's p% full, then choosing from only those words.

---

### How Top-p Works

**Example: Predicting next word**

```
Token Probabilities (sorted):
1. "sunny" → 35% (cumulative: 35%)
2. "nice" → 25% (cumulative: 60%)
3. "beautiful" → 15% (cumulative: 75%)
4. "pleasant" → 10% (cumulative: 85%)
5. "good" → 8% (cumulative: 93%)
6. "great" → 4% (cumulative: 97%)
7. "cloudy" → 2% (cumulative: 99%)
8. "rainy" → 1% (cumulative: 100%)
```

**With Top-p = 0.8 (80%):**
```
Include tokens until cumulative probability ≥ 80%:
✅ "sunny" (35%) → Total: 35%
✅ "nice" (25%) → Total: 60%
✅ "beautiful" (15%) → Total: 75%
✅ "pleasant" (10%) → Total: 85% ← Exceeds 80%!

Only these 4 tokens are considered!
(Number varies based on distribution)
```

**With Top-p = 0.95 (95%):**
```
Need more tokens to reach 95%:
✅ 6 tokens included this time

More diverse output!
```

---

### Top-p Guidelines

| Top-p Value | Behavior | Use Cases |
|-------------|----------|-----------|
| **0.1 - 0.3** | Very focused | Factual answers, code |
| **0.5 - 0.7** | Balanced | Conversation, explanations |
| **0.8 - 0.9** | Diverse | Creative writing |
| **0.95 - 1.0** | Very diverse | Experimental, brainstorming |

---

### Top-k vs Top-p

**Key Difference:**

| Aspect | Top-k | Top-p |
|--------|-------|-------|
| **Filter Size** | Fixed (always k tokens) | Dynamic (varies) |
| **When peaked distribution** | May be too restrictive | Adapts automatically |
| **When flat distribution** | May include too many | Adapts automatically |
| **Recommendation** | Use Top-p for most cases | More flexible! |

**Example:**

```
Distribution 1 (Very confident):
"Paris" → 90%, "London" → 5%, others → 5%

Top-k=10: Includes 10 tokens (unnecessary noise)
Top-p=0.9: Includes only 1 token (just "Paris") ✓ Better!

Distribution 2 (Uncertain):
"Paris" → 20%, "London" → 18%, "Berlin" → 15%, ...

Top-k=10: Includes 10 tokens ✓ Good!
Top-p=0.9: Includes ~6 tokens ✓ Also good!
```

**Top-p adapts better to different situations!**

---

## 📏 4. Max Output Tokens: Length Control

### What It Does

**Max Output Tokens** sets the **maximum number of tokens** the model can generate in its response.

**Simple Analogy:** Like setting a word limit for an essay.

---

### Important Notes

**1. Tokens ≠ Words**
```
Approximate conversion:
- 1 token ≈ 0.75 words (English)
- 100 tokens ≈ 75 words
- 1000 tokens ≈ 750 words
```

**2. Includes Input + Output** (for some models)
```
Your prompt: 50 tokens
Max tokens: 100 tokens
→ Model can generate up to 50 tokens of response

vs.

Max OUTPUT tokens: 100 tokens
→ Model can generate 100 tokens regardless of input
(Gemini uses this approach)
```

---

### Max Tokens Guidelines

| Token Count | Approximate Words | Use Cases |
|-------------|-------------------|-----------|
| **50-100** | 40-75 words | Short answers, titles |
| **200-300** | 150-225 words | Paragraph responses |
| **500-700** | 375-525 words | Detailed explanations |
| **1000-2000** | 750-1500 words | Articles, essays |
| **4000+** | 3000+ words | Long-form content |

---

### Why Limit Tokens?

**1. Cost Control**
```
More tokens = Higher API costs
Setting limits prevents unexpected bills!
```

**2. Performance**
```
Shorter responses = Faster generation
Better user experience!
```

**3. Relevance**
```
Sometimes longer ≠ better
Concise responses often more useful!
```

---

## 🛑 5. Stop Sequences: Custom Endpoints

### What It Does

**Stop Sequences** are specific strings that tell the model to **stop generating** when encountered.

**Simple Analogy:** Like a STOP sign for text generation.

---

### How Stop Sequences Work

**Example:**

```python
prompt = "List 3 fruits:\n1."
stop_sequences = ["\n4."]  # Stop before generating item 4

Output:
"1. Apple
2. Banana
3. Orange
"  ← Stops here! Doesn't generate 4.
```

---

### Common Use Cases

**1. Formatting Control**
```python
prompt = "Q: What is AI?\nA:"
stop_sequences = ["\nQ:"]  # Stop before next question

Output: "A: Artificial Intelligence is..."
(Stops before generating another Q:)
```

**2. Code Generation**
```python
prompt = "def calculate_sum():\n"
stop_sequences = ["\n\ndef ", "\nclass "]  # Stop at next function/class

Output: Complete single function only
```

**3. Dialogue Systems**
```python
stop_sequences = ["User:", "Human:"]  # Stop when user turn begins
```

---

### Multiple Stop Sequences

You can provide **multiple** stop sequences:

```python
stop_sequences = ["\n\n", "###", "END"]

Model stops when it encounters ANY of these!
```

---

## 🔁 6. Frequency Penalty: Reducing Repetition

### What It Does

**Frequency Penalty** **penalizes tokens** based on how many times they've already appeared in the generated text.

**Simple Analogy:** Like getting less excited about eating the same food over and over.

---

### How Frequency Penalty Works

**Formula:**
```
new_probability = original_probability - (frequency_penalty × token_frequency)

Where:
- token_frequency = number of times token appeared
- frequency_penalty = your setting (-2.0 to 2.0)
```

**Example:**

```
Generated so far: "The cat sat. The cat ran. The cat"

Next word predictions:
- "slept" (never used) → probability unchanged
- "jumped" (never used) → probability unchanged  
- "cat" (used 3 times) → probability REDUCED by (penalty × 3)
```

---

### Frequency Penalty Guidelines

| Value | Effect | Use Cases |
|-------|--------|-----------|
| **0.0** | No penalty (default) | General use |
| **0.1 - 0.5** | Gentle repetition reduction | Natural conversation |
| **0.6 - 1.0** | Moderate repetition reduction | Creative writing |
| **1.1 - 2.0** | Strong repetition reduction | Diverse vocabulary needed |
| **Negative** | Encourages repetition | Rarely used |

---

### Example: Effect on Output

**Prompt:** "Write about cats"

**Frequency Penalty = 0.0:**
```
"Cats are amazing. Cats love to play. Cats are independent.
Cats make great pets. Cats are popular."
(Notice: "Cats" repeated 5 times)
```

**Frequency Penalty = 1.0:**
```
"Felines are amazing. They love to play. These animals are independent.
Our furry friends make great pets. Such creatures are popular."
(Notice: Varied vocabulary, no "cats" repetition)
```

---

## 🎨 7. Presence Penalty: Encouraging New Topics

### What It Does

**Presence Penalty** penalizes tokens that have **already appeared at least once**, regardless of how many times.

**Simple Analogy:** Like encouraging someone to talk about NEW topics, not revisit old ones.

---

### Frequency Penalty vs Presence Penalty

**Key Difference:**

```
Word "cat" appears 5 times:

Frequency Penalty:
- Penalty = 0.5 × 5 = 2.5
- Scales with frequency

Presence Penalty:
- Penalty = 0.5 × 1 = 0.5
- Same penalty whether appeared 1× or 100×
```

---

### When to Use Each

| Scenario | Use Frequency Penalty | Use Presence Penalty |
|----------|----------------------|---------------------|
| **Reduce word repetition** | ✅ Better | ❌ |
| **Encourage topic diversity** | ❌ | ✅ Better |
| **Avoid repetitive phrases** | ✅ | ✅ Both work |
| **Creative brainstorming** | ❌ | ✅ Better |

---

### Presence Penalty Guidelines

| Value | Effect | Use Cases |
|-------|--------|-----------|
| **0.0** | No penalty (default) | Focused content |
| **0.1 - 0.5** | Gentle topic diversity | General writing |
| **0.6 - 1.0** | Moderate topic diversity | Brainstorming |
| **1.1 - 2.0** | Strong topic diversity | Exploring many angles |

---

### Example: Effect on Output

**Prompt:** "Discuss benefits of exercise"

**Presence Penalty = 0.0:**
```
"Exercise improves health. Exercise strengthens muscles.
Exercise boosts energy. Exercise helps weight loss.
Exercise reduces stress."
(Stays on main topic, repeats "exercise")
```

**Presence Penalty = 1.0:**
```
"Physical activity improves health. Training strengthens muscles.
Nutrition plays a role too. Mental wellness benefits from movement.
Social aspects include team sports and community."
(Explores related topics, avoids repeating words)
```

---

## 🎛️ 8. Parameter Combination Strategies

### Best Practices for Combining Parameters

**Strategy 1: Factual, Consistent Responses**
```python
temperature = 0.2
top_p = 0.3
max_output_tokens = 500
frequency_penalty = 0.0
presence_penalty = 0.0

Best for: Q&A, documentation, code generation
```

**Strategy 2: Balanced Conversation**
```python
temperature = 0.7
top_p = 0.9
max_output_tokens = 1000
frequency_penalty = 0.3
presence_penalty = 0.0

Best for: Chatbots, tutoring, general dialogue
```

**Strategy 3: Creative Content**
```python
temperature = 1.0
top_p = 0.95
max_output_tokens = 2000
frequency_penalty = 0.5
presence_penalty = 0.6

Best for: Stories, poetry, brainstorming
```

**Strategy 4: Diverse Exploration**
```python
temperature = 0.8
top_p = 0.9
max_output_tokens = 1500
frequency_penalty = 0.2
presence_penalty = 1.0

Best for: Research, multiple perspectives, ideation
```

---

## ⚠️ 9. Common Mistakes to Avoid

### Mistake 1: Conflicting Parameters
```python
❌ BAD:
temperature = 0.0  # Wants deterministic
top_p = 1.0       # Allows all tokens
# Conflict! Temperature dominates

✅ GOOD:
temperature = 0.2
top_p = 0.5
# Both work together for focused output
```

### Mistake 2: Extreme Penalties
```python
❌ BAD:
frequency_penalty = 2.0
presence_penalty = 2.0
# Output becomes unnatural, forced diversity

✅ GOOD:
frequency_penalty = 0.5
presence_penalty = 0.3
# Gentle, natural diversity
```

### Mistake 3: Wrong Temperature for Task
```python
❌ BAD:
# Generating code with high temperature
temperature = 1.5  # Code will be broken!

✅ GOOD:
temperature = 0.2  # Consistent, correct code
```

### Mistake 4: Not Setting Max Tokens
```python
❌ BAD:
# No limit set
# Could generate 10,000 tokens = high cost!

✅ GOOD:
max_output_tokens = 500  # Controlled cost
```

---

## 📊 10. Parameter Decision Tree

```
START: What kind of output do you need?
│
├─ Need EXACT, CONSISTENT answers?
│  └─ temperature=0.1, top_p=0.3, penalties=0
│
├─ Need CREATIVE, UNIQUE content?
│  └─ temperature=1.0, top_p=0.9, presence_penalty=0.6
│
├─ Need BALANCED conversation?
│  └─ temperature=0.7, top_p=0.8, frequency_penalty=0.3
│
├─ Need DIVERSE ideas/brainstorming?
│  └─ temperature=0.8, top_p=0.9, presence_penalty=1.0
│
└─ Need CODE generation?
   └─ temperature=0.2, top_p=0.5, penalties=0
```

---

## 🎓 Summary: Quick Reference Guide

| Parameter | What It Does | When to Increase | When to Decrease |
|-----------|-------------|------------------|------------------|
| **Temperature** | Randomness | Creative writing | Factual answers |
| **Top-k** | Token limit | More variety | More focus |
| **Top-p** | Dynamic filtering | Diverse output | Consistent output |
| **Max Tokens** | Length limit | Longer responses | Shorter/cheaper |
| **Frequency Penalty** | Reduce repetition | Avoid repeated words | Allow natural repetition |
| **Presence Penalty** | Topic diversity | Explore many topics | Stay on topic |

---

**Now let's see these parameters in action!** 👇

## Gemini LLM Call

In [ ]:
import os, sys
# update your project below==========>
PROJECT_ID = "dynamic-market-478415-f0"
LOCATION   = "us-central1"

# ── Authentication ─────────────────────────────────────────────────────────────
# On Colab: use built-in Google auth (no API key or service account needed)
# Locally:  set GOOGLE_APPLICATION_CREDENTIALS env var to service account JSON
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    print("✅ Authenticated via Colab")
else:
    print(f"✅ Using local credentials | Project: {PROJECT_ID}")


✅ Authenticated via Colab


# 🚀 RAG Workshop: From Basics to Advanced Techniques

## Workshop Overview

Welcome to this comprehensive hands-on workshop on **Retrieval-Augmented Generation (RAG)**!

### What You'll Learn:
1. **RAG Fundamentals** - Understanding what RAG is and why it's essential
2. **Building Your First RAG System** - Hands-on implementation from scratch


### Prerequisites:
- Basic Python knowledge
- GCP account with Vertex AI enabled
- GCP Project ID and Location

---

## 🎯 Why RAG?

### The Problem:
- LLMs have knowledge **cutoff dates**
- They can **hallucinate** (make up information)
- They don't know your **proprietary data**
- Fine-tuning is **expensive** and time-consuming
- Reduce token/limit context window

### The Solution: RAG
**Retrieval-Augmented Generation** combines:
1. **Retrieval** - Finding relevant information from your data
2. **Augmentation** - Adding that information to the prompt
3. **Generation** - LLM generates answers based on retrieved context

### Real-World Use Cases:
- 📚 **Customer Support** - Answer questions from your knowledge base
- 📄 **Document Q&A** - Query contracts, reports, manuals
- 🏢 **Enterprise Search** - Find information across company docs
- 💡 **Research Assistant** - Analyze research papers and articles
- 🎓 **Education** - Tutoring systems with course materials

---

# PDF RAG With Google ADK

This notebook builds a complete PDF-based RAG pipeline using:

- `pypdf` to read a PDF
- custom chunking logic
- Vertex AI text embeddings via `google-genai`
- ChromaDB for persistent vector storage
- a Google ADK agent that must retrieve grounded context before answering

It is set up to work both locally and in Google Colab.


## What This Does

1. Authenticates to your existing Google Cloud project.
2. Reads one or more PDFs and extracts page text.
3. Splits the text into overlapping chunks.
4. Embeds each chunk with Vertex AI.
5. Stores and retrieves chunks with ChromaDB.
6. Uses Google ADK to answer with retrieved PDF context only.


##  Roadmap

This notebook is written as a step-by-step lesson, not just a demo.

At a high level, we are building a system that does this:

1. Load PDF files.
2. Extract readable text.
3. Split that text into smaller pieces called chunks.
4. Convert each chunk into an embedding, which is a list of numbers representing meaning.
5. Store those chunks and embeddings in ChromaDB.
6. When a question arrives, embed the question too.
7. Search for the most relevant chunks.
8. Pass those chunks to a Google ADK agent so the final answer is grounded in retrieved evidence.

A useful mental model is: first we build a searchable open book, then we teach the agent how to look up the right pages before answering.

## Step 1: Install The Libraries

Each library in this notebook has one clear responsibility:

- `google-genai`: calls Gemini and the Vertex AI embedding API
- `google-adk`: provides the agent framework
- `pypdf`: reads and extracts text from PDFs
- `chromadb`: stores embeddings and performs vector search
- `numpy`: helps with vector and numeric operations

We keep embeddings in Vertex AI and storage in ChromaDB. That separation is useful because it makes the architecture easier to understand.

In [ ]:
%pip uninstall -y google-genai google-adk
%pip install -q google-adk==1.31.1 google-genai==1.73.1 pypdf numpy chromadb==1.5.8


Found existing installation: google-genai 1.73.1
Uninstalling google-genai-1.73.1:
  Successfully uninstalled google-genai-1.73.1
Found existing installation: google-adk 1.31.1
Uninstalling google-adk-1.31.1:
  Successfully uninstalled google-adk-1.31.1


## Step 2: Authenticate And Configure The Notebook

This cell sets up authentication and file paths.

- In Colab, it authenticates with the notebook user's Google account.
- Locally, it uses the `service-account-key.json` file in the workspace.

It also tells the notebook where to find PDFs and where ChromaDB should persist its local database files.

In [ ]:
import asyncio
import chromadb
import json
import os
import re
import sys
import uuid
from dataclasses import dataclass, field
from pathlib import Path

# Third-party libraries
import numpy as np
from pypdf import PdfReader
from google import genai
from google.genai import types

# Attempt to import AvatarConfig from google.generativeai.types
# and inject it into google.genai.types if missing, to satisfy google.adk.
try:
    import google.generativeai.types as _generativeai_types
    if not hasattr(types, 'AvatarConfig') and hasattr(_generativeai_types, 'AvatarConfig'):
        types.AvatarConfig = _generativeai_types.AvatarConfig
except ImportError:
    pass # google.generativeai might not be installed
except Exception as e:
    print(f"Warning: Could not patch AvatarConfig: {e}", file=sys.stderr)

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# Project and local file configuration
#Change to your project id
PROJECT_ID = 'dynamic-market-478415-f0'
LOCATION = 'us-central1'
DATA_DIR = Path('data')
PDF_PATHS = sorted(p for p in DATA_DIR.glob('*.pdf'))
SERVICE_ACCOUNT_PATH = Path('../service-account-key.json')
CHROMA_PATH = Path('data/chroma_db')

# Tell Google libraries to use Vertex AI endpoints.
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'true'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION

credentials = None
if 'google.colab' in sys.modules:
    # Colab flow: authenticate the signed-in user.
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated with Google Colab user credentials')
else:
    if SERVICE_ACCOUNT_PATH.exists():
        # Local flow: load the provided service account key.
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(SERVICE_ACCOUNT_PATH.resolve())
        from google.oauth2 import service_account
        credentials = service_account.Credentials.from_service_account_file(
            SERVICE_ACCOUNT_PATH,
            scopes=['https://www.googleapis.com/auth/cloud-platform'],
        )
        print(f'Using service account: {SERVICE_ACCOUNT_PATH.resolve()}')
    else:
        print('No local service-account-key.json found. Falling back to ADC.')

if credentials is None:
    # If we do not pass credentials explicitly, the SDK uses ADC.
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
else:
    # If we loaded a service account explicitly, pass it to the client.
    client = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=LOCATION,
        credentials=credentials,
    )

print(f'Project: {PROJECT_ID}')
print(f'Location: {LOCATION}')
print('PDF files:')
for pdf_path in PDF_PATHS:
    print(f'  - {pdf_path.resolve()}')
print(f'Chroma path: {CHROMA_PATH.resolve()}')

Authenticated with Google Colab user credentials
Project: dynamic-market-478415-f0
Location: us-central1
PDF files:
  - /content/data/employee_policy_handbook.pdf
  - /content/data/week2_rag_curriculum_excerpt.pdf
Chroma path: /content/data/chroma_db


In [ ]:
health = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Reply with exactly: Gemini call worked properly',
    config=types.GenerateContentConfig(temperature=0, max_output_tokens=100),
)


def extract_response_text(response) -> str:
    # Different SDK responses can expose text in slightly different places.
    # This helper keeps the rest of the notebook simple.
    if getattr(response, 'text', None):
        return response.text
    texts = []
    for candidate in getattr(response, 'candidates', []) or []:
        content = getattr(candidate, 'content', None)
        for part in getattr(content, 'parts', []) or []:
            if getattr(part, 'text', None):
                texts.append(part.text)
    return '\n'.join(texts)



In [ ]:
#!pip list | grep google

In [ ]:
embedding_probe = client.models.embed_content(
    model='text-embedding-004',
    contents=['RAG over PDFs with Google ADK'],
)

generation_text = extract_response_text(health).strip()
print('Generation test:', generation_text or '<empty text but request succeeded>')
print('Generation call completed')
print('Embedding dimensions:', len(embedding_probe.embeddings[0].values))

Generation test: Gemini call worked properly
Generation call completed
Embedding dimensions: 768


## Step 3: Read The PDFs

RAG begins with source documents. Here we read every PDF in the `data/` folder.

We extract text page by page because page numbers are useful later for traceability and citations.

In [ ]:
def normalize_whitespace(text: str) -> str:
    # PDF extraction often contains messy whitespace, repeated line breaks,
    # or hidden characters. This helper cleans the text into a simpler form.
    text = text.replace('\x00', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    # Return one record per page so we keep page-level metadata.
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = normalize_whitespace(page.extract_text() or '')
        if text:
            pages.append({'page_num': page_num, 'text': text, 'pdf_name': pdf_path.name})
    return pages

pages = []
for pdf_path in PDF_PATHS:
    pdf_pages = extract_pdf_pages(pdf_path)
    pages.extend(pdf_pages)
    print(f'Loaded {len(pdf_pages)} pages from {pdf_path.name}')
for page in pages[:3]:
    print(f"\n--- {page['pdf_name']} | Page {page['page_num']} preview ---")
    print(page['text'][:700])

Loaded 1 pages from employee_policy_handbook.pdf
Loaded 7 pages from week2_rag_curriculum_excerpt.pdf

--- employee_policy_handbook.pdf | Page 1 preview ---
Employee Policy Handbook This sample handbook is included so the RAG notebook can answer employee-policy questions from a realistic PDF source. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Team members must be available during core collaboration hours from 10:00 AM to 3:00 PM Central Time. Employees working remotely must use company-managed devices and connect through the approved VPN. Paid Time Off Full-time employees accrue 15 days of paid time off each calendar year. PTO requests of three or more consecutive days should be submitted at least two weeks in advance. Managers should respond to requests within five business days. Code of Condu

--- week2_rag_curriculum_excerpt.pdf | Page 1 preview ---
Week 2 RAG Curriculum Excerpt This PDF was generated from the existing Week2_3_RAG_

## Step 4: Compare Chunking Strategies

Chunking is one of the highest-impact design decisions in a RAG system.

If chunks are too small, you may lose context. If chunks are too large, retrieval can become noisy. This notebook compares three approachable strategies so you can see the tradeoffs directly.

In [ ]:
def fixed_size_chunk_text(text: str, chunk_size: int = 900, overlap: int = 120) -> list[str]:
    # Simplest strategy: fixed-size windows with overlap.
    text = text.strip()
    if not text:
        return []
    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)
    while start < len(text):
        chunk = text[start : start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

def sentence_window_chunk_text(text: str, chunk_size: int = 900, overlap_sentences: int = 1) -> list[str]:
    # Keeps sentences intact when possible.
    text = text.strip()
    if not text:
        return []
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    chunks = []
    current = []
    for sentence in sentences:
        candidate = ' '.join(current + [sentence]).strip()
        if len(candidate) <= chunk_size or not current:
            current.append(sentence)
        else:
            chunks.append(' '.join(current).strip())
            current = current[-overlap_sentences:] + [sentence]
    if current:
        chunks.append(' '.join(current).strip())
    return chunks

def recursive_paragraph_chunk_text(text: str, chunk_size: int = 900, overlap: int = 120) -> list[str]:
    # Prefers paragraph boundaries first, then falls back to sentence chunks.
    text = text.strip()
    if not text:
        return []
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if len(paragraphs) == 1:
        paragraphs = [text]
    chunks = []
    current = ''
    for paragraph in paragraphs:
        candidate = f'{current}\n\n{paragraph}'.strip() if current else paragraph
        if len(candidate) <= chunk_size:
            current = candidate
            continue
        if current:
            chunks.append(current.strip())
            tail = current[-overlap:]
        else:
            tail = ''
        if len(paragraph) <= chunk_size:
            current = f'{tail} {paragraph}'.strip()
        else:
            sentence_chunks = sentence_window_chunk_text(paragraph, chunk_size=chunk_size, overlap_sentences=1)
            if tail and sentence_chunks:
                sentence_chunks[0] = f'{tail} {sentence_chunks[0]}'.strip()
            chunks.extend(sentence_chunks[:-1])
            current = sentence_chunks[-1] if sentence_chunks else tail
    if current:
        chunks.append(current.strip())
    return [chunk for chunk in chunks if chunk]

@dataclass
class Chunk:
    # A small data object that stores text plus metadata we care about.
    text: str
    page_num: int
    chunk_id: int
    embedding: np.ndarray | None = None
    metadata: dict = field(default_factory=dict)

CHUNKING_STRATEGIES = {
    'fixed_size': fixed_size_chunk_text,
    'sentence_window': sentence_window_chunk_text,
    'recursive_paragraph': recursive_paragraph_chunk_text,
}

def build_chunks_for_strategy(pages: list[dict], strategy_name: str) -> list[Chunk]:
    # Apply one strategy across every extracted page.
    strategy_fn = CHUNKING_STRATEGIES[strategy_name]
    chunk_items = []
    for page in pages:
        page_text = page['text'].replace('. ', '.\n\n')
        for chunk_id, text in enumerate(strategy_fn(page_text), start=1):
            chunk_items.append(
                Chunk(
                    text=text,
                    page_num=page['page_num'],
                    chunk_id=chunk_id,
                    metadata={
                        'source': f"{page['pdf_name']}#page={page['page_num']}",
                        'pdf_name': page['pdf_name'],
                        'strategy': strategy_name,
                    },
                )
            )
    return chunk_items



## Step 5: Execute chunking

In [ ]:
chunk_sets = {name: build_chunks_for_strategy(pages, name) for name in CHUNKING_STRATEGIES}
for name, chunk_items in chunk_sets.items():
    avg_len = sum(len(chunk.text) for chunk in chunk_items) / max(1, len(chunk_items))
    print(f'{name}: {len(chunk_items)} chunks | average length={avg_len:.1f} chars')
    sample = chunk_items[0]
    print(sample.text[:220])
    print('-' * 80)

default_strategy = 'recursive_paragraph'
chunks = chunk_sets[default_strategy]
print(f'Default strategy for the rest of the notebook: {default_strategy}')

fixed_size: 17 chunks | average length=736.6 chars
Employee Policy Handbook This sample handbook is included so the RAG notebook can answer employee-policy questions from a realistic PDF source.

Remote Work Policy Employees may work remotely up to three days per week wi
--------------------------------------------------------------------------------
sentence_window: 19 chunks | average length=762.9 chars
Employee Policy Handbook This sample handbook is included so the RAG notebook can answer employee-policy questions from a realistic PDF source. Remote Work Policy Employees may work remotely up to three days per week wit
--------------------------------------------------------------------------------
recursive_paragraph: 18 chunks | average length=777.2 chars
Employee Policy Handbook This sample handbook is included so the RAG notebook can answer employee-policy questions from a realistic PDF source.

Remote Work Policy Employees may work remotely up to three days per week wi
--------

## Step 6: Embed And Store Chunks In ChromaDB

At this stage, plain text becomes searchable vector data.

Important distinction:
- Vertex AI creates the embeddings
- ChromaDB stores those embeddings and performs similarity search

That separation is useful because it makes the pipeline easier to reason about.

In [ ]:
class ChromaVectorStore:
    def __init__(self, client, persist_path: Path, collection_name: str, embed_model: str = 'text-embedding-004', batch_size: int = 64):
        # Each instance manages one Chroma collection.
        self.client = client
        self.persist_path = persist_path
        self.collection_name = collection_name
        self.embed_model = embed_model
        self.batch_size = batch_size
        self.persist_path.mkdir(parents=True, exist_ok=True)
        self.chroma_client = chromadb.PersistentClient(path=str(self.persist_path))
        try:
            self.chroma_client.delete_collection(name=self.collection_name)
        except Exception:
            pass
        self.collection = self.chroma_client.get_or_create_collection(name=self.collection_name)
        self.chunks: list[Chunk] = []

    def _embed_texts(self, texts: list[str], task_type: str) -> list[np.ndarray]:
        # Batch embedding is much faster than one API call per chunk.
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start : start + self.batch_size]
            response = self.client.models.embed_content(
                model=self.embed_model,
                contents=batch,
                config={'task_type': task_type},
            )
            vectors.extend(np.array(item.values) for item in response.embeddings)
        return vectors

    def add_chunks(self, chunk_items: list[Chunk]) -> None:
        # Embed documents, then store ids, documents, metadata, and vectors.
        doc_embeddings = self._embed_texts(
            [chunk.text for chunk in chunk_items],
            task_type='RETRIEVAL_DOCUMENT',
        )
        ids = []
        documents = []
        metadatas = []
        embeddings = []
        for chunk, embedding in zip(chunk_items, doc_embeddings):
            chunk.embedding = embedding
            ids.append(f"{chunk.metadata['strategy']}-{chunk.metadata['pdf_name']}-p{chunk.page_num}-c{chunk.chunk_id}")
            documents.append(chunk.text)
            metadatas.append({
                'source': chunk.metadata['source'],
                'pdf_name': chunk.metadata['pdf_name'],
                'strategy': chunk.metadata['strategy'],
                'page_num': chunk.page_num,
                'chunk_id': chunk.chunk_id,
            })
            embeddings.append(embedding.tolist())
        self.collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
        self.chunks.extend(chunk_items)

    def search(self, query: str, k: int = 4, min_score: float = 0.35) -> list[tuple[float, Chunk]]:
        # Embed the query, search Chroma, then rebuild Chunk-like results.
        # We also add a small keyword-overlap bonus so beginner-style
        # questions like 'what are embeddings used for' still surface the
        # right chunks even when vector similarity alone is imperfect.
        query_embedding = self._embed_texts([query], task_type='RETRIEVAL_QUERY')[0].tolist()
        result = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=max(k * 3, 8),
            include=['documents', 'metadatas', 'distances'],
        )
        results = []
        docs = result.get('documents', [[]])[0]
        metas = result.get('metadatas', [[]])[0]
        distances = result.get('distances', [[]])[0]
        query_terms = set(re.findall(r'[a-z0-9]+', query.lower()))
        for doc, meta, distance in zip(docs, metas, distances):
            vector_score = max(0.0, 1.0 - float(distance))
            doc_terms = set(re.findall(r'[a-z0-9]+', doc.lower()))
            keyword_overlap = len(query_terms & doc_terms) / max(1, len(query_terms))
            blended_score = (0.8 * vector_score) + (0.2 * keyword_overlap)
            results.append((blended_score, Chunk(text=doc, page_num=meta['page_num'], chunk_id=meta['chunk_id'], metadata=dict(meta))))
        results.sort(key=lambda item: item[0], reverse=True)
        strong_results = [item for item in results if item[0] >= min_score]
        if strong_results:
            return strong_results[:k]
        # Fallback: if nothing clears the threshold, still return the best
        # matches so the agent has a chance to answer from the nearest evidence.
        return results[:k]

stores = {}
for strategy_name, strategy_chunks in chunk_sets.items():
    strategy_store = ChromaVectorStore(
        client=client,
        persist_path=CHROMA_PATH / strategy_name,
        collection_name=f'pdf_rag_{strategy_name}',
    )
    strategy_store.add_chunks(strategy_chunks)
    stores[strategy_name] = strategy_store
    print(f'Indexed {len(strategy_store.chunks)} chunks into Chroma for {strategy_name}')

store = stores[default_strategy]
print(f'Using {default_strategy} Chroma collection as the default retriever for ADK')

Indexed 17 chunks into Chroma for fixed_size
Indexed 19 chunks into Chroma for sentence_window
Indexed 18 chunks into Chroma for recursive_paragraph
Using recursive_paragraph Chroma collection as the default retriever for ADK


## Step 7: Inspect Retrieval Before Adding The Agent

This is an important debugging step. If the retrieved chunks are not relevant, the agent will not be able to give a good answer either.

In [ ]:
sample_query = 'Why do we chunk documents in RAG?'
print(f'Query: {sample_query}')
for strategy_name, strategy_store in stores.items():
    print(f"\n=== Strategy: {strategy_name} ===")
    matches = strategy_store.search(sample_query, k=2)
    for rank, (score, chunk) in enumerate(matches, start=1):
        print(f"[{rank}] score={score:.4f} source={chunk.metadata['source']}")
        print(chunk.text[:350])
        print()

Query: Why do we chunk documents in RAG?

=== Strategy: fixed_size ===
[1] score=0.4766 source=week2_rag_curriculum_excerpt.pdf#page=1
Week 2 RAG Curriculum Excerpt This PDF was generated from the existing Week2_3_RAG_Basics notebook so the new RAG notebook can be tested against real local content.

■ RAG Basics — Week 2, Notebook 3 Retrieval-Augmented Generation from Scratch Welcome! This notebook teaches you **RAG (Retrieval-Augmented Generation)** — one of the most important te

[2] score=0.4201 source=week2_rag_curriculum_excerpt.pdf#page=4
Problem with whole documents:**  A 10-page document has 10 pages of text in the context window — expensive and noisy  The relevant sentence might be on page 4, but you're including all 10 pages  The AI gets confused by too much irrelevant information **Why chunks solve this:**  We find the specific chunk (maybe 100-200 words) that contains the 


=== Strategy: sentence_window ===
[1] score=0.4693 source=week2_rag_curriculum_excerpt.pdf#page=

In [ ]:
policy_query = 'How many remote days per week are allowed and what are the core hours?'
print(f'Query: {policy_query}')
policy_matches = store.search(policy_query, k=3)
for rank, (score, chunk) in enumerate(policy_matches, start=1):
    print(f"[{rank}] score={score:.4f} source={chunk.metadata['source']}")
    print(chunk.text[:350])
    print()

Query: How many remote days per week are allowed and what are the core hours?
[1] score=0.3931 source=employee_policy_handbook.pdf#page=1
Employee Policy Handbook This sample handbook is included so the RAG notebook can answer employee-policy questions from a realistic PDF source.

Remote Work Policy Employees may work remotely up to three days per week with manager approval.

Team members must be available during core collaboration hours from 10:00 AM to 3:00 PM Central Time.

Emplo



In [ ]:
def retrieve_pdf_context(question: str, top_k: int = 4) -> dict:
    """Retrieve the most relevant PDF chunks for a user question."""
    # `question` is the user's natural-language request.
    # `top_k` controls how many chunks we bring back from the vector store.
    # A smaller value is usually more precise; a larger value gives more context.
    matches = store.search(question, k=top_k)
    return {
        'question': question,
        'matches': [
            {
                'score': round(score, 4),
                'source': chunk.metadata['source'],
                'page_num': chunk.page_num,
                'chunk_id': chunk.chunk_id,
                'text': chunk.text,
            }
            for score, chunk in matches
        ],
    }

# `Agent(...)` is the core ADK object.
# It combines the model, instructions, and tools into one runnable unit.
agent = Agent(
    name='pdf_rag_agent',
    model='gemini-2.5-flash',
    instruction=(
        'You answer questions about the indexed PDF. Always call retrieve_pdf_context '
        'before answering. Use only the retrieved passages. If the passages do not '
        'contain the answer, say that clearly. Cite the source string for each claim.'
    ),
    tools=[retrieve_pdf_context],
)

# `InMemoryRunner` is a lightweight local execution engine.
# It is a great fit for notebooks because it does not require extra services.
runner = InMemoryRunner(agent=agent, app_name='pdf_rag_notebook')

async def ask_pdf(question: str) -> str:
    # Create a fresh session per question so the demo stays easy to follow.
    session_id = f'session-{uuid.uuid4().hex[:8]}'
    # `create_session(...)` prepares a conversation context for this run.
    # `app_name`, `user_id`, and `session_id` together identify the interaction.
    await runner.session_service.create_session(
        app_name='pdf_rag_notebook',
        user_id='notebook-user',
        session_id=session_id,
    )
    # `run_debug(...)` is a notebook-friendly way to execute the agent and
    # inspect the events it produced. It is especially useful for learning.
    events = await runner.run_debug(
        question,
        user_id='notebook-user',
        session_id=session_id,
        quiet=True,
    )
    final_text = ''
    for event in events:
        # We only keep the final natural-language answer.
        if event.is_final_response() and getattr(event, 'content', None):
            texts = [part.text for part in event.content.parts if getattr(part, 'text', None)]
            if texts:
                final_text = '\n'.join(texts)
    return final_text

print('ADK agent and retrieval tool are ready')

ADK agent and retrieval tool are ready


In [ ]:
questions = ['When must expense reports be submitted after a business trip?']

async def demo():
    for question in questions:
        answer = await ask_pdf(question)
        print(f'\nQuestion: {question}')
        print(answer)
        print('-' * 100)

await demo()


Question: When must expense reports be submitted after a business trip?
Expense reports must be submitted within 10 calendar days of the trip end date (source: employee_policy_handbook.pdf#page=1).
----------------------------------------------------------------------------------------------------
